<a href="https://colab.research.google.com/github/nicolasramirezperilla/DataWave-Project/blob/master/Consolidado_Balance_Valores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1) Instalar librerias y conexión al servidor.





In [ ]:
# Importación de bibliotecas
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil import parser  # Para la conversión automática de fechas
import re

# Autenticación y manejo de Google Colab
from google.colab import auth
from google.colab import files
from google.colab import data_table
auth.authenticate_user()

# Habilitación del formato de tabla para visualizar DataFrames
data_table.enable_dataframe_formatter()

# Autenticación y acceso a Google Sheets
import gspread
from google.auth import default
from oauth2client.service_account import ServiceAccountCredentials
from googleapiclient.discovery import build

# Autenticación del usuario para acceder a Google Sheets
creds, _ = default()
gc = gspread.authorize(creds)
client = gspread.authorize(creds)

#2) Descargar información & definir parametros.


In [ ]:
# Autenticación y acceso a Google Sheets
input_workbook_name = 'Consolidado Balances - Valores'
spreadsheet = gc.open(input_workbook_name)

# Obtener datos de las hojas
sheet1 = spreadsheet.worksheet("Balance_Resultado")
sheet2 = spreadsheet.worksheet("Parametria_1")
sheet4 = spreadsheet.worksheet("Inputs")
sheet5 = spreadsheet.worksheet("Parametria_2")

# Obtener fecha a actualizar y fecha anterior
fecha_actualizar = sheet4.get('B1')[0][0]  # Asumiendo que 'B1' contiene la fecha
fecha_anterior = sheet4.get('B2')[0][0]    # Asumiendo que 'B2' contiene la fecha

# ID de Google Sheet y nombre de la hoja
spreadsheet_id = '1SBs1fPPtTT-kKSgLVQlgKhhzbINA0a_ef8-svx33zRA'
sheet_name = 'Inputs'
sh = gc.open_by_key(spreadsheet_id)
worksheet = sh.worksheet(sheet_name)

# Definir rango donde se encuentran los enlaces
start_row = 5
end_row = 100  # Cambiar según la cantidad de enlaces

# Iterar sobre las celdas en la columna B para obtener y actualizar nombres de archivo
for row in range(start_row, end_row + 1):
    cell_value = worksheet.acell(f'B{row}').value  # Leer el enlace en la celda B{row}

    if cell_value:  # Verificar que la celda no esté vacía
        match = re.search(r'[-\w]{25,}', cell_value)  # Extraer el ID del archivo del enlace
        if match:
            file_id = match.group(0)
            try:
                drive_service = build('drive', 'v3', credentials=creds)
                file = drive_service.files().get(fileId=file_id).execute()
                file_name = file['name']
                worksheet.update_acell(f'C{row}', file_name)  # Escribir nombre en la celda C{row}
            except Exception:
               continue

# Función para buscar valor en la matriz
def buscar_valor(matriz, fecha_actualizar):
    for fila in matriz:
        if len(fila) >= 3 and fila[0] == fecha_actualizar:  # Asegurarse de que la fila tenga al menos tres elementos
            return fila[2]  # Retorna el valor de la columna C
    return None  # Devolver None si no se encuentra el valor

# Suponiendo que la matriz está en el rango A4:C
matriz = sheet4.get('A4:C')
input_workbook_name2 = buscar_valor(matriz, fecha_actualizar)
spreadsheet2 = gc.open(input_workbook_name2)

# Obtener datos y construir DataFrames
# DataFrame desde Balance_Resultado
data1_range = sheet1.get_all_values()
column_names = data1_range[0]
data = data1_range[1:]
df_temp = pd.DataFrame(data, columns=column_names)

# Seleccionar y renombrar columnas en df1
columna_inicio = 'cuenta'
indice_inicio = df_temp.columns.get_loc(columna_inicio)
indice_final = df_temp.columns.get_loc(fecha_anterior)
df1 = df_temp.iloc[:, indice_inicio:indice_final + 1]
df1 = df1.rename(columns={'Cuenta': 'cuenta', 'Nombre_Cuenta': 'nombre_cuenta'})

# DataFrame desde Parametria_1
data2_range = sheet2.get('A:H')
df2 = pd.DataFrame(data2_range[1:], columns=data2_range[0])
df2['Nucta'] = df2['Nucta'].str.replace(',', '').astype(int)
df2 = df2.rename(columns={'Descripcion': 'nombre_cuenta 1'})

# DataFrame desde Parametria_2
data22_range = sheet5.get('A:G')
df22 = pd.DataFrame(data22_range[1:], columns=data22_range[0])
df22['Nucta'] = df22['Nucta'].str.replace(',', '').astype(int)

# DataFrame desde la hoja Sheet1 del segundo archivo
sheet3 = spreadsheet2.worksheet('Sheet1')
data4_range = sheet3.get('A:H')
df4 = pd.DataFrame(data4_range[1:], columns=data4_range[0])
df4 = df4.rename(columns={'Cuenta': 'cuenta 1', 'Nombre_Cuenta': 'nombre_cuenta 1'})

#3) Cruce bases.
Cruces:
1. Parameterias
2. Balance Primario: Historico / Balance Mensual
3. Balance Secundario: Balance Primario / Parametrias

In [ ]:
merged_df1 = df2.rename(columns={'Nucta': 'cuenta 1','Descripción': 'nombre_cuenta 1'})
df1['cuenta'] = df1['cuenta'].astype(str)
merged_df1['cuenta 1'] = merged_df1['cuenta 1'].astype(str)
df4['cuenta 1'] = df4['cuenta 1'].astype(str)

merged_df2 = pd.merge(merged_df1, df1, left_on="cuenta 1", right_on="cuenta", how="right")
columnas_a_borrar = ['cuenta 1', 'nombre_cuenta 1']
merged_df2 = merged_df2.drop(columns=columnas_a_borrar, errors='ignore')

merged_df3 = pd.merge(merged_df2, df4[['cuenta 1', 'nombre_cuenta 1', 'Saldo_Final']],left_on="cuenta", right_on="cuenta 1", how="left")
merged_df3 = merged_df3.drop(columns=columnas_a_borrar, errors='ignore')
merged_df3 = merged_df3.rename(columns={'Saldo_Final': fecha_actualizar})

#4) Base de Nuevas Cuentas + Concantenado Balance.

In [ ]:
merged_df44 = pd.merge(df4[['cuenta 1', 'nombre_cuenta 1', 'Saldo_Final']], merged_df2[['cuenta']], left_on="cuenta 1", right_on="cuenta", how='left', indicator=True)
merged_df44 = merged_df44[merged_df44['_merge'] == 'left_only'].drop(columns=['_merge'])
merged_df44 = merged_df44.drop_duplicates(subset=['cuenta 1', 'nombre_cuenta 1'], keep=False)
merged_df44 = merged_df44.drop(columns='cuenta', errors='ignore')
merged_df44 = merged_df44.rename(columns={'cuenta 1': 'cuenta','nombre_cuenta 1': 'nombre_cuenta','Saldo_Final':'valor'})
merged_df44['cuenta'] = merged_df44['cuenta'].fillna('0')
merged_df44['cuenta'] = merged_df44['cuenta'].str.replace(',', '').astype(int)
merged_df44['valor'] = merged_df44['valor'].str.replace('.', '').str.replace(',', '').astype(int)
merged_df44['Parameteria'] = merged_df44['cuenta'].apply(lambda x: any(str(x) in str(y) for y in df2['Nucta']))
merged_df44['Min_Lvl'] = merged_df44['cuenta'].apply(lambda x: not any(merged_df44['cuenta'].astype(str).str.startswith(str(x)) & (merged_df44['cuenta'] != x)))

merged_df4 = merged_df44.rename(columns={'valor': fecha_actualizar})

# Concatenar DataFrames
merged_df_concat = pd.concat([merged_df3, merged_df4], ignore_index=True)
merged_df_concat = merged_df_concat.drop(columns=['Parameteria', 'Min_Lvl','Nombre'])
merged_df_concat = merged_df_concat.drop_duplicates(subset=['cuenta', 'nombre_cuenta'])

#5) Formato + Clasificaciones.

*  Formato de número
*  Clasificacion tipo (B,P&L,O)
*  Renombramiento columnas
*  Reordenamiento columnas
*  Creación filas "detalle"
*  Clasificacion codigo (001,002,003)
*  Clasificacion minimo lvl (TRUE,FALSE)
*  Interpretación valores NaN

In [ ]:
def formato_solo_enteros(valor):
    try:
        valor = int(valor)  # Intentar convertir a entero
        return valor
    except (TypeError, ValueError):
        if isinstance(valor, str) and ',' in valor:
            # Si es una cadena y contiene una coma, eliminar todo después de la coma
            valor = valor.split(',')[0]
        return valor  # Devolver el valor modificado o el valor original si no se puede convertir

# Función para convertir el formato numérico
def convert_to_numeric(value):
    try:
        if isinstance(value, str):
            # Cuenta puntos y comas en el valor
            num_puntos = value.count('.')
            num_comas = value.count(',')

            # Decide el formato en función de la cantidad
            if num_puntos > num_comas:
                # Más puntos que comas
                return float(value.replace('.', '').replace(',', '.'))
            else:
                # Más comas que puntos
                return float(value.replace(',', ''))
        # Si ya es un número, lo convierte directamente a float
        return float(value)
    except ValueError:
        return None # Retorna None en caso de error de conversión

# Obtener columnas a llenar
columns_to_fill = merged_df_concat.columns[merged_df_concat.columns.get_loc('nombre_cuenta') + 1:]
merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].fillna(value=0)

# Aplica la conversión a la columna especificada
merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].applymap(convert_to_numeric)
merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].applymap(formato_solo_enteros)

filtered_df = merged_df_concat[merged_df_concat['CM'].str.contains('Impuesto Sociedades', na=False)]
filtered_df.head()

<ipython-input-12-2dd14d1e8e31>:36: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].applymap(convert_to_numeric)
<ipython-input-12-2dd14d1e8e31>:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].applymap(formato_solo_enteros)


,CM,CM Detalle,Neocon,Neocon Detalle,Gestión,Gestión Detalle,cuenta,nombre_cuenta,31/12/2022,31/01/2023,...,29/02/2024,31/03/2024,30/04/2024,31/05/2024,30/06/2024,31/07/2024,31/08/2024,30/09/2024,31/10/2024,30/11/2024
965,Impuesto Sociedades,A la Renta,"36,501",IMPUESTOS SOBRE BENEFICIOS,"49,840",IMPUESTO SOCIEDADES,57050501,Impuesto de renta y complementarios,0,165675856,...,393059204,818297779,2732115901,3183032244,4991500750,7440494573,7980284471,10087122228,10495670963,9890376821
966,Impuesto Sociedades,A la Renta,"36,501",IMPUESTOS SOBRE BENEFICIOS,"49,840",IMPUESTO SOCIEDADES,57050501,Impuesto de renta y complementarios + detalle,0,165675856,...,393059204,818297779,2732115901,3183032244,4991500750,7440494573,7980284471,10087122228,10495670963,9890376821
967,Impuesto Sociedades,Impuesto diferido,"36,505",IMPUESTOS SOBRE BENEFICIOS,"49,840",IMPUESTO SOCIEDADES,57050504,Impuesto Diferido + detalle,0,-100379172,...,456906267,-148340056,-85337237,125150773,-101021648,-51215582,16590474,-116723900,-249626712,-668779382
968,Impuesto Sociedades,Impuesto diferido,"36,505",IMPUESTOS SOBRE BENEFICIOS,"49,840",IMPUESTO SOCIEDADES,57050504,Impuesto Diferido,0,-100379172,...,319621964,171281908,85944671,211095444,110073796,58858214,75448688,-41275212,-290901924,-668779382
969,Impuesto Sociedades,Sobretasa 5%,"36,505",IMPUESTOS SOBRE BENEFICIOS,"49,840",IMPUESTO SOCIEDADES,57050505,Tarifa Especial Entidades Ficieras + detalle,0,0,...,0,0,390302271,64416621,258352643,349856261,77112842,300976823,58364105,1412910974


In [ ]:
#---------------------
# Obtener columnas de fechas
fechas = [col for col in merged_df_concat.columns if '/' in col]

# Filtrar cuentas que comienzan con 4 o 5
df_filtered = merged_df_concat[merged_df_concat['cuenta'].astype(str).str.startswith(('4', '5'))]

# Crear un DataFrame vacío para las nuevas filas
new_rows = []

for _, row in df_filtered.iterrows():
    cuenta = row['cuenta']
    nombre_cuenta = row['nombre_cuenta']

    detalle_name = f"{nombre_cuenta} + detalle"

    # Verificar si ya existe una fila con "+ detalle"
    existing_detalle_row = merged_df_concat[merged_df_concat['nombre_cuenta'] == detalle_name]

    if existing_detalle_row.empty:
        # Crear la fila con "+ detalle"
        new_row = {'cuenta': cuenta, 'nombre_cuenta': detalle_name}

        # Calcular las diferencias mes a mes
        prev_value = row[fechas[0]]
        differences = [0]  # La diferencia para el primer mes es 0

        for fecha in fechas[1:]:
            current_value = row[fecha]
            difference = current_value - prev_value
            differences.append(difference)
            prev_value = current_value

        # Añadir las diferencias al nuevo DataFrame
        new_row.update(dict(zip(fechas, differences)))
        new_rows.append(new_row)

    else:
        # Actualizar la fila existente con las nuevas diferencias
        existing_row_index = existing_detalle_row.index[0]
        prev_value = row[fechas[0]]

        updated_differences = [0]  # La diferencia para el primer mes es 0

        for fecha in fechas[1:]:
            current_value = row[fecha]
            difference = current_value - prev_value
            updated_differences.append(difference)
            prev_value = current_value

        merged_df_concat.loc[existing_row_index, fechas] = updated_differences

# Convertir las nuevas filas a un DataFrame y concatenar con el original
new_df = pd.DataFrame(new_rows)
merged_df_concat = pd.concat([merged_df_concat, new_df], ignore_index=True)

# Asegurarse de que no haya filas duplicadas para "+ detalle"
#merged_df_concat = merged_df_concat.drop_duplicates(subset=['cuenta',fecha_actualizar], keep='last')
# Eliminar filas con más de un "+ detalle"
def remove_extra_detalle(name):
    if name.count(' + detalle') > 1:
        return None
    return name

merged_df_concat['nombre_cuenta'] = merged_df_concat['nombre_cuenta'].apply(remove_extra_detalle)
merged_df_concat = merged_df_concat.dropna(subset=['nombre_cuenta'])
#---------------------------------------------------------------------------------------------------
#merged_df_concat = merged_df_concat.replace('nan', '', regex=True)
merged_df_concat['cuenta'] = merged_df_concat['cuenta'].astype(str)

# Definir la función para categorizar cuentas
def categorizar_cuenta(cuenta):
    cuenta_str = str(cuenta) if pd.notna(cuenta) else ''
    if cuenta_str.startswith(('1', '2', '3')):
        return 'B'
    elif cuenta_str.startswith(('4', '5')):
        return 'P&L'
    else:
        return 'O'

# Aplicar categorización de cuentas
merged_df_concat['TIPO'] = merged_df_concat['cuenta'].apply(categorizar_cuenta)
columnas = ['TIPO'] + [col for col in merged_df_concat if col != 'TIPO']
merged_df_concat = merged_df_concat[columnas]

# Renombrar las columnas del DataFrame

original_columns = ['TIPO', 'LOCAL', 'LOCAL II', 'COD CONSOLIDACION', 'COD DE GESTION', 'NOMBRE CONSO', 'NOMBRE GESTION']
new_columns = ['TIPO', 'CM', 'CM Detalle', 'Neocon', 'Neocon Detalle', 'Gestión', 'Gestión Detalle']
column_mapping = dict(zip(original_columns, new_columns))
merged_df_concat = merged_df_concat.rename(columns=column_mapping)

# Obtener columnas a llenar
columns_to_fill = merged_df_concat.columns[merged_df_concat.columns.get_loc('nombre_cuenta') + 1:]
merged_df_concat[columns_to_fill] = merged_df_concat[columns_to_fill].fillna(value=0)

# Encuentra la posición de la columna 'nombre_cuenta'
nombre_cuenta_index = merged_df_concat.columns.get_loc('nombre_cuenta')

merged_df_concat_new=merged_df_concat

# Definir la función para asignar valores a la nueva columna
def assign_code(row):
    nombre_cuenta_str = str(row['nombre_cuenta'])
    cuenta_str = str(row['cuenta'])
    if 'detalle' in nombre_cuenta_str:
        return '003'
    elif cuenta_str.startswith(('1', '2', '3')):
        return '001'
    elif cuenta_str.startswith(('4', '5')):
        return '002'
    else:
        return ''

# Crear la nueva columna con valores basados en la condición
merged_df_concat_new['Clasificacion'] = merged_df_concat_new.apply(assign_code, axis=1)

merged_df_concat_new['Min_Lvl'] = merged_df_concat['cuenta'].apply(lambda x: not any(merged_df_concat['cuenta'].str.startswith(x)& (merged_df_concat['cuenta']!=x)))

# Reordenar las columnas para colocar 'new_column' antes de 'cuenta'
columns = merged_df_concat_new.columns.tolist()
cuenta_index = columns.index('cuenta')

# Insertar 'new_column' antes de 'cuenta'
columns.insert(cuenta_index, columns.pop(columns.index('Clasificacion')))
merged_df_concat_new = merged_df_concat_new[columns]

# Reordenar las columnas para colocar 'Min_Lvl' antes de 'cuenta'
columns = merged_df_concat_new.columns.tolist()
cuenta_index = columns.index('cuenta')

# Insertar 'Min_Lvl' antes de 'cuenta'
columns.insert(cuenta_index, columns.pop(columns.index('Min_Lvl')))
merged_df_concat_new = merged_df_concat_new[columns]

merged_df_concat_new[columns_to_fill] = merged_df_concat_new[columns_to_fill].fillna(value=0)
merged_df_concat_new[columns_to_fill] = merged_df_concat_new[columns_to_fill].applymap(formato_solo_enteros)

merged_df_concat_new = merged_df_concat_new.sort_values(by='cuenta', ascending=True)

merged_df_concat_new['cuenta'] = merged_df_concat_new['cuenta'].str.replace(',', '').astype(float)

# Si deseas restablecer los índices después de ordenar
merged_df_concat_new.reset_index(drop=True, inplace=True)

<ipython-input-13-e9d21550f1f5>:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df_concat['cuenta'] = merged_df_concat['cuenta'].astype(str)
<ipython-input-13-e9d21550f1f5>:82: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df_concat['TIPO'] = merged_df_concat['cuenta'].apply(categorizar_cuenta)
<ipython-input-13-e9d21550f1f5>:137: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df_concat_new[columns_to_fill] = merged_df_concat_new[columns_to_fill].a

#6) Actualizar hojas de cálculo en Google Sheets.

In [ ]:
# Actualizar hojas de cálculo en Google Sheets
output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Balance_Resultado')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([merged_df_concat_new.columns.values.tolist()] + merged_df_concat_new.fillna('').values.tolist())

output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Nuevas_Cuentas')
existing_data = output_sheet_balance_resultado.get_all_values()
if 'fecha_actualizar' not in merged_df44.columns:
    merged_df44['fecha_actualizar'] = fecha_actualizar
header = merged_df44.columns.tolist()
new_data = merged_df44.fillna('').values.tolist()
if existing_data:
    combined_data = existing_data[1:] + new_data
else:
    combined_data = new_data
output_sheet_balance_resultado.update([header] + combined_data)

sheet6 = spreadsheet.worksheet("Nuevas_Cuentas")
data6_range = sheet6.get('A:F')
df66 = pd.DataFrame(data6_range[1:], columns=data6_range[0])
df66['cuenta'] = df66['cuenta'].str.replace(',', '').astype(int)
df66['valor'] = df66['valor'].str.replace('.', '').str.replace(',', '').astype(int)
df66['validacion'] = df66['cuenta'].isin(merged_df_concat_new['cuenta'])
df66 = df66.sort_values(by='fecha_actualizar')
df66 = df66.drop_duplicates(subset='cuenta', keep='first')
df66 = df66[df66['Parameteria'] != 'TRUE']


output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Nuevas_Cuentas')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([df66.columns.values.tolist()] + df66.fillna('').values.tolist())

output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Parametria_1')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([df2.columns.values.tolist()] + df2.fillna('').values.tolist())

#df2 = df22.rename(columns={'Nucta': 'cuenta'})
output_sheet_balance_resultado = gc.open(input_workbook_name).worksheet('Parametria_2')
output_sheet_balance_resultado.clear()
output_sheet_balance_resultado.update([df22.columns.values.tolist()] + df22.fillna('').values.tolist())

{'spreadsheetId': '1SBs1fPPtTT-kKSgLVQlgKhhzbINA0a_ef8-svx33zRA',
 'updatedRange': 'Parametria_2!A1:G722',
 'updatedRows': 722,
 'updatedColumns': 7,
 'updatedCells': 5054}